# Tema 1 · Redes neuronales

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

**Agrícola Los Maquis** exporta arándanos desde San Carlos. En el packing, el control de calidad mide cada muestra de fruta (calibre, firmeza y azúcar) y decide si es **apta para exportar**. El jefe de packing quiere un modelo que haga esa clasificación automáticamente con las tres mediciones. Ya probaron una regresión y «no funcionó».

> **❓ La pregunta del cliente:** ¿Puede un modelo decidir con las tres mediciones qué fruta es apta para exportar, y por qué la regresión no pudo?

## 🗂️ Los datos: `t1_arandanos.csv` (600 muestras de fruta con calibre, firmeza, °Brix y si fue apta)

| Columna | Qué significa |
|---|---|
| `muestra` | número de la muestra |
| `calibre_mm` | diámetro del fruto, en milímetros |
| `firmeza` | resistencia al apretar (más alto = más duro) |
| `brix` | azúcar, en grados Brix |
| `apta` | **Sí / No**: si el control de calidad la aprobó para exportar ← lo que se quiere predecir |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t1_arandanos.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 600 filas y 5 columnas.

## 1. Conocer los datos

📘 Cada fila es una **muestra de fruta** de un lote de arándanos, medida en el packing: el **calibre** (diámetro en
mm), la **firmeza** (cuánto resiste al apretarla) y los **°Brix** (azúcar). La columna `apta` dice si el control de
calidad la aprobó para exportar. La celda cuenta cuántas muestras hay de cada clase.

In [ ]:
datos.head()

In [ ]:
conteo = datos["apta"].value_counts()
print("Aptas:", conteo["Sí"], "· No aptas:", conteo["No"])
# ✅ Debe mostrar: Aptas: 201 · No aptas: 399

## 2. Mirar el problema antes de modelar

📘 El gráfico ubica cada muestra según su **firmeza** (eje x) y sus **°Brix** (eje y). Verde = apta; gris = no apta.
Miren **dónde** quedan las aptas.

In [ ]:
aptas = datos[datos["apta"] == "Sí"]
no_aptas = datos[datos["apta"] == "No"]
plt.figure(figsize=(6, 4.5))
plt.scatter(no_aptas["firmeza"], no_aptas["brix"], s=12, c="lightgray", label="No apta")
plt.scatter(aptas["firmeza"], aptas["brix"], s=12, c="green", label="Apta")
plt.xlabel("Firmeza"); plt.ylabel("°Brix (azúcar)"); plt.legend()
plt.title("¿Dónde están las muestras aptas?")
plt.show()

### ✍️ ¿Dónde quedan las muestras aptas en el gráfico? ¿Se podrían separar de las no aptas con **una sola línea recta**? ¿Por qué?

_Su respuesta:_

## 3. Separar datos de entrenamiento y de prueba

📘 Para saber si un modelo sirve de verdad, se evalúa con muestras **que no vio** al aprender. Se separa el 75 %
para **entrenar** y el 25 % para **probar** (`stratify` mantiene la misma proporción de aptas en ambos).

In [ ]:
X = datos[["calibre_mm", "firmeza", "brix"]]
y = datos["apta"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, random_state=2026, stratify=y)
print("Entrenamiento:", len(X_ent), "muestras · Prueba:", len(X_pru), "muestras")
# ✅ Debe mostrar: Entrenamiento: 450 muestras · Prueba: 150 muestras

## 4. La línea base: ZeroR

📘 Antes de cualquier modelo, la pregunta es: ¿cuánto se acierta **sin pensar**? ZeroR dice siempre la clase más
común. Cualquier modelo que valga la pena debe superar este número.

In [ ]:
acierto_zeror = (y_pru == "No").mean()
print(f"ZeroR (decir siempre 'No'): {acierto_zeror:.0%}")
# ✅ Debe mostrar: ZeroR (decir siempre 'No'): 67%

## 5. Un modelo que solo traza una recta: regresión logística

📘 La **regresión logística** es la «prima» clasificadora de la regresión lineal: combina las columnas con pesos y
traza **una frontera recta** entre aptas y no aptas. Es, en el fondo, **una sola neurona**. La función
`dibujar_frontera()` pinta de verde la zona donde el modelo diría «apta» (con calibre de 16 mm).

In [ ]:
def dibujar_frontera(modelo, titulo):
    """Pinta la zona que el modelo clasifica como 'apta' (calibre fijo en 16 mm) y encima las muestras de prueba."""
    f, b = np.meshgrid(np.linspace(120, 320, 200), np.linspace(8, 18, 200))
    grilla = pd.DataFrame({"calibre_mm": 16, "firmeza": f.ravel(), "brix": b.ravel()})
    zona = (modelo.predict(grilla) == "Sí").reshape(f.shape)
    plt.figure(figsize=(6, 4.5))
    plt.contourf(f, b, zona, levels=[-0.5, 0.5, 1.5], colors=["white", "#c8e6c9"])
    colores = np.where(y_pru == "Sí", "green", "gray")
    plt.scatter(X_pru["firmeza"], X_pru["brix"], s=14, c=colores)
    plt.xlabel("Firmeza"); plt.ylabel("°Brix (azúcar)"); plt.title(titulo)
    plt.show()

recta = LogisticRegression(max_iter=1000).fit(X_ent, y_ent)
acierto_recta = recta.score(X_pru, y_pru)
print(f"Regresión logística (una recta): {acierto_recta:.0%}")
dibujar_frontera(recta, "Una sola recta")
# ✅ Debe mostrar: Regresión logística (una recta): 66%

### ✍️ Comparen el acierto de la regresión logística con el de ZeroR. ¿Aprendió algo útil? Usen el gráfico de la frontera para explicar por qué.

_Su respuesta:_

## 6. La red neuronal

📘 Una **neurona** artificial hace tres cosas: (1) recibe las entradas (calibre, firmeza, °Brix), (2) multiplica
cada una por un **peso** y las suma, y (3) pasa esa suma por una **función de activación** que decide cuánto «se
enciende». Una neurona sola traza una recta. Una **red** pone varias neuronas en una **capa oculta**: cada una
traza su propia recta y la neurona de salida las **combina**, y así se puede dibujar una curva cerrada.

- `StandardScaler()` pone las tres columnas en la misma escala (la red aprende mal si una columna va de 120 a 320 y otra de 8 a 18).
- `MLPClassifier(hidden_layer_sizes=(8,))` es una red con **una capa oculta de 8 neuronas**.
- `max_iter` es el máximo de **vueltas de entrenamiento**: en cada vuelta la red compara sus respuestas con las
  correctas y ajusta los pesos un poco para equivocarse menos.

In [ ]:
red = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(8,), max_iter=3000, random_state=1))
red.fit(X_ent, y_ent)
acierto_red = red.score(X_pru, y_pru)
print(f"Red neuronal (8 neuronas): {acierto_red:.0%}")
print("¿Le gana a la recta por más de 15 puntos?", acierto_red - acierto_recta > 0.15)
dibujar_frontera(red, "Red neuronal con 8 neuronas")
# ✅ Debe mostrar: Red neuronal (8 neuronas): 87%  /  ¿Le gana a la recta por más de 15 puntos? True

### ✍️ ¿Cuánto acierta la red y cómo es su frontera comparada con la de la recta? ¿Qué hizo la red que la regresión logística no podía hacer?

_Su respuesta:_

## 7. ¿Cuántas neuronas hacen falta?

📘 El número de neuronas de la capa oculta es una decisión del analista. Con muy pocas, la red no alcanza a
dibujar la isla; con muchas, se vuelve más lenta y más difícil de entender, y puede aprenderse de memoria los
datos de entrenamiento.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
NEURONAS = 8        # ✏️ pruebe con 1, 2, 4, 8 y 32 (al terminar, deje 8)

prueba_red = make_pipeline(StandardScaler(),
                           MLPClassifier(hidden_layer_sizes=(NEURONAS,), max_iter=3000, random_state=1))
prueba_red.fit(X_ent, y_ent)
print(f"{NEURONAS} neuronas → acierto en prueba: {prueba_red.score(X_pru, y_pru):.0%}")
dibujar_frontera(prueba_red, f"Red con {NEURONAS} neuronas")

**✏️ Anoten aquí sus resultados:**

| Neuronas | Acierto en prueba | ¿Le gana a la recta? |
|:-:|:-:|:-:|
| 1 | | |
| 2 | | |
| 4 | | |
| 8 | | |
| 32 | | |

### ✍️ Según su tabla, ¿desde cuántas neuronas la red supera claramente a la recta? ¿Qué le pasa a la frontera con 1 neurona y con 32? ¿Cuántas recomendarían y por qué?

_Su respuesta:_

## 8. Mirar dentro de la red: ¿se puede explicar?

📘 Todo lo que la red «sabe» está en sus **pesos**: números que se ajustaron al entrenar. La celda los cuenta.

In [ ]:
mlp = red[-1]                       # la red, sin el escalador
pesos = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Números que aprendió la red:", pesos)
print("Pesos entre las 3 entradas y las 8 neuronas ocultas:", mlp.coefs_[0].shape)
# ✅ Debe mostrar: Números que aprendió la red: 41  /  Pesos entre las 3 entradas y las 8 neuronas ocultas: (3, 8)

### ✍️ La red aprendió 41 números. ¿Podrían explicarle al jefe de packing, con esos números, **por qué** una muestra es apta? Comparen con un árbol de decisión o con OneR.

_Su respuesta:_

## 9. Usar el modelo con muestras nuevas

📘 Tres muestras recién medidas, todas de 16 mm. `predict_proba` entrega la **probabilidad** de que cada una sea apta.

In [ ]:
nuevas = pd.DataFrame({"calibre_mm": [16, 16, 16],
                       "firmeza":    [220, 300, 150],
                       "brix":       [13.0, 9.0, 17.0]})
predicciones = red.predict(nuevas)
probabilidades = red.predict_proba(nuevas)[:, list(red.classes_).index("Sí")]
for (_, fila), pred, prob in zip(nuevas.iterrows(), predicciones, probabilidades):
    print(f"Firmeza {fila['firmeza']:.0f}, °Brix {fila['brix']:.0f} → {pred} (prob. de apta: {prob:.1f})")
# ✅ Debe mostrar: Firmeza 220, °Brix 13 → Sí (prob. de apta: 1.0)  /  Firmeza 300, °Brix 9 → No (prob. de apta: 0.0)  /  Firmeza 150, °Brix 17 → No (prob. de apta: 0.1)

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para Agrícola Los Maquis: ¿conviene usar la red para separar la fruta en el packing? ¿Qué ganarían, qué riesgos tiene y qué harían con la fruta que la red rechaza?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema1.ipynb`, junto con el informe en PDF.